# Debug an OpenAI Agents workflow with AgentDbg

This tutorial shows how to trace and debug a multi-step OpenAI Agents SDK workflow using AgentDbg. It uses the Agents tracing API to emit deterministic spans, so it runs without API keys or network calls.

## Why agent debugging is hard

AI agents run multi-step workflows with LLM calls, tools, and handoffs between components. When something goes wrong, it is unclear which step failed or why. AgentDbg captures a structured timeline of every run so you can see exactly what happened.

## Setup

If not done already, install AgentDbg with OpenAI Agents support and the OpenAI Agents SDK (run once):

In [9]:
# %pip install --upgrade pip -q
# %pip install "agentdbg[openai]" "openai-agents" -q

In [10]:
import os

from agentdbg import trace, AgentDbgLoopAbort
from agentdbg.integrations import openai_agents

from agents.tracing import (
    function_span,
    generation_span,
    handoff_span,
    set_trace_processors,
    trace as agents_trace,
)

## Happy-path OpenAI Agents trace

We will model the same **quarterly sales** workflow as the LangGraph tutorial:

- **LLM** decides to search for quarterly sales.
- **Tool** returns deterministic sales data.
- **LLM** decides to calculate the total.
- **Tool** returns the total.
- **LLM** decides to save the result.
- **Tool** confirms saving.
- **LLM** returns a final DONE summary.

Instead of building a graph, we emit OpenAI Agents tracing spans that represent these steps. The AgentDbg OpenAI Agents adapter translates them into `LLM_CALL` and `TOOL_CALL` events.

In [11]:
@trace(name="OpenAI Agents (happy path)")
def run_openai_agents_happy_path():
    # Tell the OpenAI Agents SDK to send tracing spans to AgentDbg.
    set_trace_processors([openai_agents.PROCESSOR])

    # This block simulates an OpenAI Agents run with deterministic spans.
    with agents_trace("AgentDbg OpenAI Agents happy path"):
        # Agent decides to search.
        with generation_span(
            input=[{"role": "user", "content": "Get quarterly sales and save the total."}],
            output=[{"role": "assistant", "content": "I will search for quarterly sales."}],
            model="gpt-4o-mini",
            model_config={"temperature": 0.0},
            usage={"prompt_tokens": 10, "completion_tokens": 10, "total_tokens": 20},
        ):
            pass

        # search tool returns deterministic quarterly sales.
        with function_span(
            name="search",
            input={"query": "quarterly sales"},
            output={"data": "Q1: 1.2M, Q2: 1.5M, Q3: 2.0M, Q4: 1.8M"},
        ):
            pass

        # Agent decides to calculate the total.
        with generation_span(
            input=[{"role": "assistant", "content": "I will search for quarterly sales."}],
            output=[{"role": "assistant", "content": "I will calculate the total."}],
            model="gpt-4o-mini",
            model_config={"temperature": 0.0},
            usage={"prompt_tokens": 8, "completion_tokens": 8, "total_tokens": 16},
        ):
            pass

        # calculator tool returns the numeric total.
        with function_span(
            name="calculator",
            input={"expression": "1.2+1.5+2.0+1.8"},
            output={"result": 6.5},
        ):
            pass

        # Agent decides to save the result.
        with generation_span(
            input=[{"role": "assistant", "content": "I will calculate the total."}],
            output=[{"role": "assistant", "content": "I will save the result."}],
            model="gpt-4o-mini",
            model_config={"temperature": 0.0},
            usage={"prompt_tokens": 8, "completion_tokens": 8, "total_tokens": 16},
        ):
            pass

        # save_result tool confirms saving.
        with function_span(
            name="save_result",
            input={"data": 6.5},
            output={"result": "Saved: 6.5"},
        ):
            pass

        # Agent returns DONE summary.
        with generation_span(
            input=[{"role": "assistant", "content": "I will save the result."}],
            output=[{"role": "assistant", "content": "DONE. Summary: 6.5M total."}],
            model="gpt-4o-mini",
            model_config={"temperature": 0.0},
            usage={"prompt_tokens": 8, "completion_tokens": 12, "total_tokens": 20},
        ):
            pass


run_openai_agents_happy_path()
print("Run complete. View with: agentdbg view")

Run complete. View with: agentdbg view


In [12]:
!agentdbg list

run_id  	run_name                                 	started_at              	duration_ms	llm_calls	tool_calls	status
5aa07674	OpenAI Agents (happy path)               	2026-03-17T21:52:57.530Z	77         	4        	3         	ok    
54021c15	OpenAI Agents (looping, with guardrail)  	2026-03-17T21:52:25.186Z	94         	3        	3         	error 
2f6016ff	OpenAI Agents (looping, no guardrail)    	2026-03-17T21:52:24.497Z	356        	26       	25        	ok    
fd2d4d63	OpenAI Agents (happy path)               	2026-03-17T21:52:24.080Z	81         	4        	3         	ok    
9d4b31f0	OpenAI Agents (looping, with guardrail)  	2026-03-17T21:38:08.690Z	365        	26       	25        	error 
762195bd	OpenAI Agents (looping, no guardrail)    	2026-03-17T21:38:08.025Z	354        	26       	25        	ok    
cabb6bd5	OpenAI Agents (happy path)               	2026-03-17T21:38:07.602Z	82         	4        	3         	ok    
efc5404b	OpenAI Agents (looping, with guardrail)  	2026-03-17T21:34:52.5

## View the timeline

In a **terminal**, run:

```bash
agentdbg view
```

A browser opens at `http://127.0.0.1:8712`. You will see:

- **Sidebar**: Recent runs (newest first) with name, time, status, duration.
- **Run summary**: Status (OK/ERROR/RUNNING), counts (LLM calls, tools, errors, loop warnings), and filter chips (All, LLM, Tools, Errors, State, Loops).
- **Timeline**: Events in order; expand any event to see payload and meta as JSON.

For this run, the OpenAI Agents adapter turned each **generation span** into an `LLM_CALL` and each **function span** into a `TOOL_CALL`. The meta section contains OpenAI Agents-specific details under `meta.openai_agents.*`. 

### What happened in this run

1. **RUN_START** — Trace began.
2. **LLM_CALL** — Agent decided to search.
3. **TOOL_CALL** (`search`) — Search returned quarterly figures.
4. **LLM_CALL** — Agent decided to calculate.
5. **TOOL_CALL** (`calculator`) — Total 6.5.
6. **LLM_CALL** — Agent decided to save.
7. **TOOL_CALL** (`save_result`) — Saved.
8. **LLM_CALL** — Agent said DONE.
9. **RUN_END** — Trace finished successfully.

This is the same logical sequence as the LangGraph tutorial, but driven by OpenAI Agents tracing spans instead of a graph.

## The looping workflow

Next we simulate an agent **stuck in a loop**: the LLM keeps saying "search again" so the system keeps calling the search tool and re-asking the LLM. We set AgentDbg's loop-detection env vars so a repeated pattern is detected quickly.

In [13]:
os.environ.setdefault("AGENTDBG_LOOP_WINDOW", "6")
os.environ.setdefault("AGENTDBG_LOOP_REPETITIONS", "3")

'3'

In [14]:
@trace(name="OpenAI Agents (looping, no guardrail)")
def run_openai_agents_looping_no_guardrail():
    set_trace_processors([openai_agents.PROCESSOR])

    with agents_trace("AgentDbg OpenAI Agents looping (no guardrail)"):
        # LLM keeps deciding to search again several times.
        for _ in range(5):
            with generation_span(
                input=[{"role": "user", "content": "Find sales"}],
                output=[{"role": "assistant", "content": "I will search again."}],
                model="gpt-4o-mini",
                model_config={"temperature": 0.0},
                usage={"prompt_tokens": 6, "completion_tokens": 6, "total_tokens": 12},
            ):
                pass

            with function_span(
                name="search",
                input={"query": "quarterly sales"},
                output={"data": "Q1: 1.2M, Q2: 1.5M, Q3: 2.0M, Q4: 1.8M"},
            ):
                pass

        # Eventually the agent decides it is done.
        with generation_span(
            input=[{"role": "assistant", "content": "I will search again."}],
            output=[{"role": "assistant", "content": "DONE."}],
            model="gpt-4o-mini",
            model_config={"temperature": 0.0},
            usage={"prompt_tokens": 4, "completion_tokens": 4, "total_tokens": 8},
        ):
            pass


run_openai_agents_looping_no_guardrail()
print("Run finished (no stop_on_loop). Check the timeline for LOOP_WARNING.")

Run finished (no stop_on_loop). Check the timeline for LOOP_WARNING.


In [15]:
!agentdbg list

run_id  	run_name                                 	started_at              	duration_ms	llm_calls	tool_calls	status
7b03f315	OpenAI Agents (looping, no guardrail)    	2026-03-17T21:52:57.943Z	115        	6        	5         	ok    
5aa07674	OpenAI Agents (happy path)               	2026-03-17T21:52:57.530Z	77         	4        	3         	ok    
54021c15	OpenAI Agents (looping, with guardrail)  	2026-03-17T21:52:25.186Z	94         	3        	3         	error 
2f6016ff	OpenAI Agents (looping, no guardrail)    	2026-03-17T21:52:24.497Z	356        	26       	25        	ok    
fd2d4d63	OpenAI Agents (happy path)               	2026-03-17T21:52:24.080Z	81         	4        	3         	ok    
9d4b31f0	OpenAI Agents (looping, with guardrail)  	2026-03-17T21:38:08.690Z	365        	26       	25        	error 
762195bd	OpenAI Agents (looping, no guardrail)    	2026-03-17T21:38:08.025Z	354        	26       	25        	ok    
cabb6bd5	OpenAI Agents (happy path)               	2026-03-17T21:38:07.6

## Understanding LOOP_WARNING

AgentDbg's loop detector looks at the **last N events** (the "window"). If the same **signature pattern** (for example, a `TOOL_CALL` named `search` followed by an `LLM_CALL` from the same provider) repeats **3 times** in a row, it emits a **LOOP_WARNING** event.

In the viewer, use the **Loops** filter to see it; the event payload shows the pattern and which event IDs formed the evidence.

Here the workflow called **search** and the **LLM** three times in a row with no progress. AgentDbg flags that so you can fix the prompt or logic instead of burning more tokens.

## Fix with stop_on_loop

To **prevent** a runaway loop, use the **stop_on_loop** guardrail. When a LOOP_WARNING is emitted and the repetition count meets the minimum, AgentDbg raises **AgentDbgLoopAbort**.

**Key difference from LangChain**: the OpenAI Agents SDK wraps all tracing-processor callbacks in `try/except` and swallows exceptions. There is no `raise_error` flag to force propagation. That means `AgentDbgLoopAbort` **cannot stop the SDK from inside a processor callback**. The adapter stores the exception on `PROCESSOR.abort_exception` instead.

To actually stop the loop you must **check `PROCESSOR.abort_exception` inside your loop** and break early. After the loop, call `PROCESSOR.raise_if_aborted()` so the `@trace` context records the proper `ERROR` and `RUN_END(status="error")`.

In [16]:
from agentdbg.integrations.openai_agents import PROCESSOR


@trace(
    name="OpenAI Agents (looping, with guardrail)",
    stop_on_loop=True,
    stop_on_loop_min_repetitions=3,
)
def run_openai_agents_looping_with_guardrail():
    set_trace_processors([openai_agents.PROCESSOR])

    with agents_trace("AgentDbg OpenAI Agents looping (with guardrail)"):
        for i in range(5):
            with generation_span(
                input=[{"role": "user", "content": "Find sales"}],
                output=[{"role": "assistant", "content": "I will search again."}],
                model="gpt-4o-mini",
                model_config={"temperature": 0.0},
                usage={"prompt_tokens": 6, "completion_tokens": 6, "total_tokens": 12},
            ):
                pass

            with function_span(
                name="search",
                input={"query": "quarterly sales"},
                output={"data": "Q1: 1.2M, Q2: 1.5M, Q3: 2.0M, Q4: 1.8M"},
            ):
                pass

            # The SDK swallows exceptions from tracing processors, so we
            # must check for the abort ourselves and break out of the loop.
            if PROCESSOR.abort_exception is not None:
                print(f"  Loop stopped early at iteration {i + 1}")
                break

    PROCESSOR.raise_if_aborted()


try:
    run_openai_agents_looping_with_guardrail()
    print("Unexpected: run did not abort")
except AgentDbgLoopAbort as e:
    print(f"AgentDbg stopped the workflow: {e}")
    print("The full trace is saved; open it with: agentdbg view")

Error in trace processor <agentdbg.integrations.openai_agents.AgentDbgOpenAIAgentsTracingProcessor object at 0x7203bf718440> during on_span_end: guardrail stop_on_loop: repetitions 3 >= stop_on_loop_min_repetitions 3


  Loop stopped early at iteration 3
AgentDbg stopped the workflow: guardrail stop_on_loop: repetitions 3 >= stop_on_loop_min_repetitions 3
The full trace is saved; open it with: agentdbg view


The loop was **stopped early** after iteration 3 (the minimum needed to detect the pattern), not all 25. The timeline contains all events up to and including the `LOOP_WARNING`, so you can see exactly which spans formed the loop.

Notice the difference from the LangGraph tutorial: with LangChain, the callback handler sets `raise_error = True` so the graph stops automatically. With the OpenAI Agents SDK, you must poll `PROCESSOR.abort_exception` yourself. In a real `Runner.run()` call, you would check it after the runner returns rather than inside the loop.

## Next steps

- Run `agentdbg view` in your terminal to explore the traces from this notebook.
- See the main README for more guardrails, storage, and integrations.
- Try the examples: `examples/openai_agents/minimal.py` and the LangGraph tutorial notebook to compare traces side by side.